# Model Comparison and Analysis

This notebook compares different machine learning models for fraud detection.

## Objectives
- Train multiple ML models (XGBoost, LightGBM, Random Forest)
- Compare model performance metrics
- Analyze feature importance across models
- Select best model for production

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

# ML models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

# XGBoost and LightGBM
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available")

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available")

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

print(f"XGBoost available: {XGBOOST_AVAILABLE}")
print(f"LightGBM available: {LIGHTGBM_AVAILABLE}")

## 1. Load and Prepare Data

In [ ]:
# Load engineered features
data_path = '../data/processed/engineered_features.csv'

try:
    df = pd.read_csv(data_path)
    print(f"Loaded engineered features: {df.shape}")
except FileNotFoundError:
    print("No engineered features found. Generating sample data...")
    
    # Generate sample data for demonstration
    np.random.seed(42)
    n_samples = 5000
    
    df = pd.DataFrame({
        'amount': np.random.exponential(100, n_samples),
        'amount_log': np.random.exponential(4, n_samples),
        'hour': np.random.randint(0, 24, n_samples),
        'day_of_week': np.random.randint(0, 7, n_samples),
        'merchant_category': np.random.randint(1, 10, n_samples),
        'customer_age': np.random.randint(18, 80, n_samples),
        'customer_tenure': np.random.randint(0, 365, n_samples),
        'is_online': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
        'is_foreign': np.random.choice([0, 1], n_samples, p=[0.9, 0.1]),
        'distance_from_home': np.random.exponential(50, n_samples),
        'is_night': np.random.choice([0, 1], n_samples, p=[0.7, 0.3]),
        'is_weekend': np.random.choice([0, 1], n_samples, p=[0.7, 0.3]),
        'is_high_amount': np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
        'risk_score': np.random.randint(0, 5, n_samples),
        'is_fraud': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
    })
    
    print(f"Generated sample data: {df.shape}")

# Prepare features and target
if 'is_fraud' in df.columns:
    X = df.drop('is_fraud', axis=1)
    y = df['is_fraud']
else:
    # Assume last column is target
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

print(f"Features shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"Fraud rate: {y.mean():.3f}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Training fraud rate: {y_train.mean():.3f}")
print(f"Test fraud rate: {y_test.mean():.3f}")

## 2. Define Models

In [ ]:
# Define models to compare
models = {}

# Random Forest
models['Random Forest'] = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

# Logistic Regression
models['Logistic Regression'] = LogisticRegression(
    random_state=42,
    class_weight='balanced',
    max_iter=1000
)

# XGBoost
if XGBOOST_AVAILABLE:
    scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc'
    )

# LightGBM
if LIGHTGBM_AVAILABLE:
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        class_weight='balanced',
        verbose=-1
    )

print(f"Models to compare: {list(models.keys())}")

## 3. Train and Evaluate Models

In [ ]:
# Train and evaluate each model
results = {}
predictions = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    if name in ['XGBoost', 'LightGBM']:
        # Tree-based models don't need scaled features
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    else:
        # Use scaled features for other models
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        if hasattr(model, 'predict_proba'):
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        else:
            y_pred_proba = y_pred  # Fallback
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba),
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }
    
    results[name] = metrics
    predictions[name] = {'y_pred': y_pred, 'y_pred_proba': y_pred_proba}
    
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1_score']:.4f}")
    print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")

## 4. Compare Model Performance

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame(results).T
print("Model Comparison:")
print(comparison_df.round(4))

# Plot comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']

for i, metric in enumerate(metrics_to_plot):
    row, col = i // 3, i % 3
    comparison_df[metric].sort_values().plot(kind='barh', ax=axes[row, col])
    axes[row, col].set_title(f'{metric.replace("_", " ").title()}')
    axes[row, col].set_xlabel('Score')
    
# Remove empty subplot
axes[1, 2].remove()

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves comparison
plt.figure(figsize=(10, 8))

for name, pred in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, pred['y_pred_proba'])
    auc_score = results[name]['roc_auc']
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})', lw=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Analyze feature importance for tree-based models
feature_importance = {}

for name, model in models.items():
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        feature_importance[name] = dict(zip(X.columns, importance))
        
        # Plot top 10 features
        plt.figure(figsize=(10, 6))
        indices = np.argsort(importance)[::-1][:10]
        plt.bar(range(10), importance[indices])
        plt.title(f'Top 10 Feature Importance - {name}')
        plt.xlabel('Features')
        plt.ylabel('Importance')
        plt.xticks(range(10), [X.columns[i] for i in indices], rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

# Compare feature importance across models
if feature_importance:
    # Create DataFrame for comparison
    importance_df = pd.DataFrame(feature_importance).fillna(0)
    
    # Plot average importance
    avg_importance = importance_df.mean(axis=1).sort_values(ascending=False)
    
    plt.figure(figsize=(12, 8))
    avg_importance.head(15).plot(kind='bar')
    plt.title('Average Feature Importance Across Models')
    plt.xlabel('Features')
    plt.ylabel('Average Importance')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print("Top 15 Features by Average Importance:")
    print(avg_importance.head(15))

## 6. Cross-Validation Analysis

In [ ]:
# Perform cross-validation for more robust evaluation
cv_results = {}
cv_folds = 5

for name, model in models.items():
    print(f"\nCross-validating {name}...")
    
    if name in ['XGBoost', 'LightGBM']:
        # Use original features
        scores = cross_val_score(model, X, y, cv=cv_folds, scoring='roc_auc')
    else:
        # Use scaled features
        X_scaled = scaler.fit_transform(X)
        scores = cross_val_score(model, X_scaled, y, cv=cv_folds, scoring='roc_auc')
    
    cv_results[name] = {
        'mean_score': scores.mean(),
        'std_score': scores.std(),
        'scores': scores
    }
    
    print(f"  Mean ROC-AUC: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")
    print(f"  Scores: {scores}")

# Plot cross-validation results
plt.figure(figsize=(10, 6))
means = [cv_results[name]['mean_score'] for name in cv_results.keys()]
stds = [cv_results[name]['std_score'] for name in cv_results.keys()]
names = list(cv_results.keys())

plt.bar(names, means, yerr=stds, capsize=5, alpha=0.7)
plt.title('Cross-Validation ROC-AUC Scores')
plt.xlabel('Models')
plt.ylabel('ROC-AUC Score')
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Model Selection Summary

In [ ]:
# Create comprehensive summary
summary_data = []

for name in models.keys():
    if name in results and name in cv_results:
        summary_data.append({
            'Model': name,
            'Test Accuracy': results[name]['accuracy'],
            'Test Precision': results[name]['precision'],
            'Test Recall': results[name]['recall'],
            'Test F1-Score': results[name]['f1_score'],
            'Test ROC-AUC': results[name]['roc_auc'],
            'CV ROC-AUC': cv_results[name]['mean_score'],
            'CV Std': cv_results[name]['std_score'],
            'Fraud Detected': int(results[name]['confusion_matrix'][1][1]),
            'False Positives': int(results[name]['confusion_matrix'][0][1]),
            'False Negatives': int(results[name]['confusion_matrix'][1][0])
        })

summary_df = pd.DataFrame(summary_data)

print("=== MODEL COMPARISON SUMMARY ===")
print(summary_df.round(4))

# Find best model by different criteria
best_auc = summary_df.loc[summary_df['Test ROC-AUC'].idxmax()]
best_f1 = summary_df.loc[summary_df['Test F1-Score'].idxmax()]
best_recall = summary_df.loc[summary_df['Test Recall'].idxmax()]
most_stable = summary_df.loc[summary_df['CV Std'].idxmin()]

print("\n=== BEST MODELS BY CRITERIA ===")
print(f"Best ROC-AUC: {best_auc['Model']} ({best_auc['Test ROC-AUC']:.4f})")
print(f"Best F1-Score: {best_f1['Model']} ({best_f1['Test F1-Score']:.4f})")
print(f"Best Recall: {best_recall['Model']} ({best_recall['Test Recall']:.4f})")
print(f"Most Stable: {most_stable['Model']} (CV std: {most_stable['CV Std']:.4f})")

# Overall recommendation (weighted score)
summary_df['Overall Score'] = (
    summary_df['Test ROC-AUC'] * 0.4 +
    summary_df['Test F1-Score'] * 0.3 +
    summary_df['Test Recall'] * 0.2 +
    (1 - summary_df['CV Std']) * 0.1  # Lower std is better
)

best_overall = summary_df.loc[summary_df['Overall Score'].idxmax()]
print(f"\nRecommended Model: {best_overall['Model']} (Score: {best_overall['Overall Score']:.4f})")

## 8. Save Results

In [ ]:
# Save comparison results
import json
from datetime import datetime

# Prepare results for saving
results_to_save = {
    'timestamp': datetime.now().isoformat(),
    'dataset_info': {
        'total_samples': len(X),
        'training_samples': len(X_train),
        'test_samples': len(X_test),
        'fraud_rate': y.mean(),
        'features': list(X.columns)
    },
    'model_comparison': summary_df.to_dict('records'),
    'detailed_metrics': results,
    'cross_validation': cv_results,
    'feature_importance': feature_importance,
    'recommendation': {
        'best_overall_model': best_overall['Model'],
        'best_overall_score': float(best_overall['Overall Score']),
        'best_auc_model': best_auc['Model'],
        'best_auc_score': float(best_auc['Test ROC-AUC'])
    }
}

# Save to file
output_path = '../data/processed/model_comparison_results.json'
os.makedirs('../data/processed', exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(results_to_save, f, indent=2, default=str)

print(f"Results saved to: {output_path}")

# Save summary CSV
summary_csv_path = '../data/processed/model_comparison_summary.csv'
summary_df.to_csv(summary_csv_path, index=False)
print(f"Summary CSV saved to: {summary_csv_path}")

## 9. Conclusions

### Key Findings:

1. **Performance Ranking**: Models ranked by overall performance score
2. **Stability**: Cross-validation results show model consistency
3. **Feature Importance**: Key features identified across models
4. **Trade-offs**: Balance between precision, recall, and stability

### Recommendations:

1. **Production Model**: Use the model with highest overall score
2. **Monitoring**: Track model performance drift over time
3. **Ensemble**: Consider combining multiple models for better performance
4. **Threshold Tuning**: Optimize decision thresholds based on business requirements

### Next Steps:

1. Deploy recommended model to production
2. Set up monitoring and alerting
3. Schedule regular model retraining
4. Implement A/B testing for new models